# Marker-subset informativeness

**Role:** Analysis.

Compare saved all-marker and lineage/subset checkpoints across depth on matched organoids. Feature order and subset membership come from the run, and cell-specific evaluation uses original raw markers, even when a trained model uses a reduced panel. This replaces separate depth and regional marker-subset reports.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Select run and models

Use the model catalog to select complete fold/seed families. For modern training runs, each catalog row also records marker subset and graph-signal treatment.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'lineage_removal_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'marker_informativeness'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Evaluate recorded validation data

Compute predictions using the saved transformations, then average squared errors within each organoid. Cells are not treated as independent biological replicates.

In [ ]:
MODEL_KEYS = run.records.key.tolist()  # Checkpoint keys included in the comparison.
frames = []
for key in MODEL_KEYS:
    chosen = run.select(key, device=DEVICE)
    frame = prediction_table(chosen, device=DEVICE)
    record = run.records.set_index('key').loc[key]
    for column in ['name','depth','fold','seed','subset','signal']:
        frame[column] = record[column]
    frame['hidden_dim'] = record.get('hidden_dim', chosen['model'].hidden_dim)
    frame['mask_rate'] = str(record.get('rate', 'none'))
    frame['key'] = key
    frames.append(frame)
cells = pd.concat(frames, ignore_index=True)
keys = ['key','name','depth','fold','seed','subset','signal','hidden_dim','mask_rate','organoid_str','n_cells']
organoids = cells.groupby(keys).squared_error.mean().rename('mse').reset_index()
if 'baseline_squared_error' in cells:
    baseline_scores = cells.groupby(keys).baseline_squared_error.mean().rename('baseline_mse').reset_index()
    organoids = organoids.merge(baseline_scores, on=keys, validate='one_to_one')
    organoids['mse_minus_baseline'] = organoids.mse-organoids.baseline_mse
    display(organoids.groupby(['name','subset','signal','hidden_dim','mask_rate','depth'])[
        ['mse','baseline_mse','mse_minus_baseline']].mean())
else:
    print('This historical run does not contain saved baseline predictions.')
summary = organoids.groupby(['name','subset','signal','hidden_dim','mask_rate','depth']).mse.agg(['mean','std','count'])
display(summary)
organoids.to_csv(OUTPUT_DIR/'organoid_mse.csv',index=False)

## Compare depth and organoid size

Lines summarize the same saved families; size bins are descriptive. Seed/fold dispersion is shown in tables rather than being presented as independent biological confidence intervals.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
organoids['size_bin'] = pd.qcut(organoids.n_cells,5,duplicates='drop')
for label, frame in organoids.groupby(['name','subset','signal','hidden_dim','mask_rate']):
    depth = frame.groupby('depth').mse.mean()
    line, = axes[0].plot(depth.index,depth.values,'o-',label=str(label))
    if 'baseline_mse' in frame:
        baseline_depth = frame.groupby('depth').baseline_mse.mean()
        axes[0].plot(baseline_depth.index,baseline_depth.values,'--',color=line.get_color(),label=f'{label}: baseline')
    by_n=frame.groupby('size_bin',observed=True).agg(N=('n_cells','median'),mse=('mse','mean'))
    line, = axes[1].plot(by_n.N,by_n.mse,'o-',label=str(label))
    if 'baseline_mse' in frame:
        baseline_n = frame.groupby('size_bin',observed=True).agg(N=('n_cells','median'),mse=('baseline_mse','mean'))
        axes[1].plot(baseline_n.N,baseline_n.mse,'--',color=line.get_color(),label=f'{label}: baseline')
axes[0].set(xlabel='Depth / radius',ylabel='Mean organoid MSE')
axes[1].set(xscale='log',xlabel='Cell count N',ylabel='Mean organoid MSE')
axes[0].legend(fontsize=7);plt.tight_layout();plt.show()
display(organoids.groupby(['name','subset','signal','depth','fold','seed']).mse.mean())

## Error among cells expressing each original marker

Score each model on the same original marker-positive validation cells. Coexpressing cells can enter more than one marker group; labels are never inferred from predictions.

In [ ]:
marker_rows=[]
for key in MODEL_KEYS:
    chosen=run.select(key,device=DEVICE)
    names=chosen.get('all_marker_names',chosen['marker_names'])
    for org, frame in cells[cells.key==key].groupby('organoid_str'):
        original=chosen['raw_graphs'][org].x.numpy()
        for j,marker in enumerate(names):
            selected=frame[original[frame.node.to_numpy(),j]>.5]
            if len(selected):
                marker_rows.append(dict(key=key,marker=marker,organoid_str=org,mse=selected.squared_error.mean()))
marker_scores=pd.DataFrame(marker_rows)
display(marker_scores.groupby(['key','marker']).mse.agg(['mean','count']))
marker_scores.to_csv(OUTPUT_DIR/'marker_mse.csv',index=False)

## Optional regional subset comparison

Join saved node-level predictions to circumference-qualified anatomical labels. Compare marker-subset errors on the same regions and organoids, retaining annotation-unavailable cells separately. This uses the same regional definitions as region_predictability, while permitting reduced-marker models.

In [ ]:
COMPARE_REGIONS = False  # Compare saved model errors by anatomical region.
if COMPARE_REGIONS:
    REGION_TABLE = ROOT/'results_experiments/ki67_observed_neighborhoods/exclusive_v2_profile_necks/nodes.csv.gz'  # Node-keyed anatomical annotation table; use qualified necks.
    regions=pd.read_csv(REGION_TABLE,usecols=['organoid_str','node','region'])
    regional=cells.merge(regions,on=['organoid_str','node'],how='left',validate='many_to_one')
    regional['region']=regional.region.fillna('annotation_unavailable')
    regional_scores=regional.groupby(['key','organoid_str','region']).squared_error.mean().rename('mse').reset_index()
    display(regional_scores.groupby(['key','region']).mse.agg(['mean','count']))
    regional_scores.to_csv(OUTPUT_DIR/'regional_subset_mse.csv',index=False)